# Data Wrangling — Synthetic Vietnamese Students’ Feedback

Nguồn: [Kaggle — toreleon/synthetic-vietnamese-students-feedback-corpus](https://www.kaggle.com/datasets/toreleon/synthetic-vietnamese-students-feedback-corpus)

1. `student_feedback_data/synthetic_train.csv` (train)  
2. `student_feedback_data/synthetic_val.csv` (validation)  

Dữ liệu **văn bản**: missing → làm sạch câu; chuẩn hoá đơn vị → điểm sentiment; normalize → độ dài câu; binning → ngắn/trung/dài; dummy → `topic`.

Mỗi bước có **1 dòng mẫu**. Dòng `# TODO` em làm tương tự (đổi tên cột / file).

## Download data and explore

Luôn bắt đầu bằng **nhìn dữ liệu**, đừng nhảy vào `fillna`. Hỏi: bao nhiêu dòng? cột nào object? câu rỗng / chỉ khoảng trắng?

| Cột | Kiểu | Ý nghĩa |
|---|---|---|
| `sentence` | object | Câu phản hồi (VN hoặc EN). Không trùng trong từng file. |
| `sentiment` | object | Nhãn cảm xúc: `negative` / `neutral` / `positive` (gần cân bằng). |
| `topic` | object | Chủ đề: `lecturer`, `curriculum`, `facility`, `others`. |

**Ghi chú:** Không có `NaN` sẵn. Một số câu là tiếng Anh (không dấu). Cần tạo biến số (độ dài câu) để normalize / binning.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

path_train = 'student_feedback_data/synthetic_train.csv'
path_val = 'student_feedback_data/synthetic_val.csv'

df_train = pd.read_csv(path_train)
df_val = pd.read_csv(path_val)

print('Train shape:', df_train.shape)
print('Val shape:', df_val.shape)

print('\n--- Train dtypes ---')
print(df_train.dtypes)

print('\n--- Train Head ---')
display(df_train.head())

print('\n--- Val Head ---')
display(df_val.head())


---
# Bộ 1 — Train

## 1. Xử lý giá trị thiếu (Missing Values)

Câu rỗng / chỉ khoảng trắng → `NaN`. `sentiment` / `topic` (phân loại) → **mode**. `sentence` là nội dung chính → **drop** dòng thiếu rồi `reset_index`.

In [ ]:
df = df_train.copy()

df['sentence'] = df['sentence'].astype(str).str.strip()
df['sentiment'] = df['sentiment'].astype(str).str.strip()
df['topic'] = df['topic'].astype(str).str.strip()

for col in ['sentence', 'sentiment', 'topic']:
    df[col] = df[col].replace({'nan': np.nan, '': np.nan, 'None': np.nan})

print('Missing values trước xử lý (Train):')
print(df.isnull().sum())

df['sentiment'] = df['sentiment'].fillna(df['sentiment'].mode(dropna=True)[0])
df['topic'] = df['topic'].fillna(df['topic'].mode(dropna=True)[0])

df = df.dropna(subset=['sentence']).reset_index(drop=True)

print('\nMissing values sau xử lý (Train):')
print(df.isnull().sum())
print('Số dòng còn lại:', len(df))


## 2. Sửa định dạng dữ liệu (Correct Data Format)

Ép `sentiment`, `topic` về kiểu `category`. Thêm độ dài câu (`n_chars`, `n_words`) dạng số.

In [ ]:
print('Kiểu dữ liệu ban đầu:\n', df.dtypes)

df['sentiment'] = df['sentiment'].astype('category')
df['topic'] = df['topic'].astype('category')

df['n_chars'] = df['sentence'].str.len().astype(int)
df['n_words'] = df['sentence'].str.split().str.len().astype(int)

print('\nKiểu dữ liệu sau khi ép:\n', df.dtypes)
df[['sentence', 'sentiment', 'topic', 'n_chars', 'n_words']].head()


## 3. Chuẩn hoá dữ liệu (Data Standardization)

Đưa nhãn cảm xúc về **cùng thang số**: `negative → -1`, `neutral → 0`, `positive → 1`.

In [ ]:
sentiment_map = {'negative': -1, 'neutral': 0, 'positive': 1}
df['sentiment_score'] = df['sentiment'].map(sentiment_map).astype(int)

df[['sentiment', 'sentiment_score']].drop_duplicates()


## 4. Chuẩn hoá phạm vi giá trị (Data Normalization)

`n_chars` lớn hơn `n_words` → `x / x.max()` về 0–1.

In [ ]:
df['n_chars_normalized'] = df['n_chars'] / df['n_chars'].max()
df['n_words_normalized'] = df['n_words'] / df['n_words'].max()

df[['n_chars', 'n_chars_normalized', 'n_words', 'n_words_normalized']].head()


## 5. Phân nhóm (Binning)

Chia độ dài câu (số từ) thành Short / Medium / Long bằng `pd.cut()`.

In [ ]:
bins = np.linspace(df['n_words'].min(), df['n_words'].max(), 4)

df['length_binned'] = pd.cut(
    df['n_words'], bins=bins, labels=['Short', 'Medium', 'Long'], include_lowest=True
)

print('Số lượng câu theo khoảng độ dài (Train):\n', df['length_binned'].value_counts())

plt.figure(figsize=(8, 5))
plt.hist(df['n_words'], bins=3, color='mediumseagreen', edgecolor='black')
plt.xlabel('Số từ trong câu (n_words)')
plt.ylabel('Tần suất (Frequency)')
plt.title('Phân bố độ dài câu phản hồi (Train Set)')
plt.show()

df[['n_words', 'length_binned']].head()


## 6. Biến chỉ thị / Biến giả (Dummy Variable)

`topic` (chữ) → mỗi chủ đề một cột 0/1. Giữ `sentiment` làm nhãn dự đoán.

In [ ]:
dummy_topic = pd.get_dummies(df['topic'], prefix='topic', dtype=int)
df = pd.concat([df, dummy_topic], axis=1)
df = df.drop(columns=['topic'])

df.head()


## Check and save

Trước khi `to_csv`:

* `isnull().sum()` còn 0 (hoặc đúng chỗ cố ý để thiếu)
* `n_chars`, `n_words` là số
* có cột mới: `sentiment_score`, `*_normalized`, `length_binned`, `topic_*`

`index=False` để file không thêm cột index 0,1,2,…


In [ ]:
df.to_csv('student_feedback_train_clean.csv', index=False)
print('Kích thước tập Train sạch:', df.shape)
df_train_clean = df.copy()


---
---
# Bộ 2 — Validation

Làm **giống bộ train**, đổi `df_train` → `df_val`. Copy mẫu ở trên, sửa tên biến / title. Kết thúc bằng **Check and save** (`to_csv`, `index=False`).


In [ ]:
# Bộ 2 — Validation Set: Thực hiện đầy đủ 6 bước
df = df_val.copy()

# Bước 1: Missing values
df['sentence'] = df['sentence'].astype(str).str.strip()
df['sentiment'] = df['sentiment'].astype(str).str.strip()
df['topic'] = df['topic'].astype(str).str.strip()

for col in ['sentence', 'sentiment', 'topic']:
    df[col] = df[col].replace({'nan': np.nan, '': np.nan, 'None': np.nan})

df['sentiment'] = df['sentiment'].fillna(df['sentiment'].mode(dropna=True)[0])
df['topic'] = df['topic'].fillna(df['topic'].mode(dropna=True)[0])
df = df.dropna(subset=['sentence']).reset_index(drop=True)

# Bước 2: Ép kiểu & độ dài
df['sentiment'] = df['sentiment'].astype('category')
df['topic'] = df['topic'].astype('category')
df['n_chars'] = df['sentence'].str.len().astype(int)
df['n_words'] = df['sentence'].str.split().str.len().astype(int)

# Bước 3: Sentiment score
sentiment_map = {'negative': -1, 'neutral': 0, 'positive': 1}
df['sentiment_score'] = df['sentiment'].map(sentiment_map).astype(int)

# Bước 4: Normalization
df['n_chars_normalized'] = df['n_chars'] / df['n_chars'].max()
df['n_words_normalized'] = df['n_words'] / df['n_words'].max()

# Bước 5: Binning
bins_val = np.linspace(df['n_words'].min(), df['n_words'].max(), 4)
df['length_binned'] = pd.cut(
    df['n_words'], bins=bins_val, labels=['Short', 'Medium', 'Long'], include_lowest=True
)
print('Số lượng câu theo khoảng độ dài (Validation):\n', df['length_binned'].value_counts())

plt.figure(figsize=(8, 5))
plt.hist(df['n_words'], bins=3, color='goldenrod', edgecolor='black')
plt.xlabel('Số từ trong câu (n_words)')
plt.ylabel('Tần suất (Frequency)')
plt.title('Phân bố độ dài câu phản hồi (Validation Set)')
plt.show()

# Bước 6: Dummy variables
dummy_topic_val = pd.get_dummies(df['topic'], prefix='topic', dtype=int)
df = pd.concat([df, dummy_topic_val], axis=1)
df = df.drop(columns=['topic'])

# Save clean validation dataset
df.to_csv('student_feedback_val_clean.csv', index=False)
print('Kích thước tập Validation sạch:', df.shape)
df_val_clean = df.copy()
df.head()
